[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghiridharravi-athenatec/claude_code_flow/blob/main/run_in_colab.ipynb)

# Clinical Documentation Integrity Scorer -- Run in Colab (ngrok tunnel)

Clones the Clinical Documentation Integrity Scorer repo, installs backend + frontend dependencies, builds the frontend, starts one combined server (API + frontend, one port), and exposes it publicly via a single [ngrok](https://ngrok.com) tunnel so you can use the app from a normal browser tab.

**Before running:** you'll need a free ngrok account and its authtoken (https://dashboard.ngrok.com/get-started/your-authtoken) -- you'll be prompted for it below. You'll also need your own Anthropic (Claude) API key ready, but you paste that into the *app's* UI once it's running, never into this notebook.

Run the cells in order, top to bottom. This notebook assumes a Linux environment with Node.js/npm and git already available (true of the standard Colab runtime).

## 1. Configuration

In [ ]:
# TODO: replace with your repository's URL
REPO_URL = "https://github.com/ghiridharravi-athenatec/claude_code_flow.git"
BRANCH = "main"
REPO_DIR = "cdi-scorer-repo"

# One port, one process, one ngrok tunnel: the Flask backend serves both the
# API and the built frontend (see Step 7), so there's nothing to configure
# for a separate frontend port.
BACKEND_PORT = 5000

## 2. Clone the repository

In [ ]:
import os

if "<your-username>" in REPO_URL:
    raise ValueError("Set REPO_URL in the Configuration cell above to your actual repository URL before continuing.")

if os.path.isdir(REPO_DIR):
    print(f"'{REPO_DIR}' already exists -- skipping clone. Delete the folder and re-run this cell to re-clone.")
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

REPO_PATH = os.path.abspath(REPO_DIR)
BACKEND_DIR = os.path.join(REPO_PATH, "backend")
FRONTEND_DIR = os.path.join(REPO_PATH, "frontend")

print("Repo:    ", REPO_PATH)
print("Backend: ", BACKEND_DIR)
print("Frontend:", FRONTEND_DIR)

## 3. Install backend dependencies

In [ ]:
# libmagic is the native library python-magic wraps (SPEC.md Section 1.1) --
# Colab's Debian base doesn't ship it by default.
!apt-get -qq update && apt-get install -y -qq libmagic1

!pip install -q -r {BACKEND_DIR}/requirements.txt

print("Backend dependencies installed.")

Optional: pre-download the spaCy model used for PHI redaction on the backend's unhandled-error logging path (~400MB). Not required for the normal upload -> score flow -- skip unless you specifically want it warmed up in advance.

In [ ]:
# !python -m spacy download en_core_web_lg

## 4. Install frontend dependencies

In [ ]:
!npm install --prefix {FRONTEND_DIR}

## 5. Build the frontend for production

Built now, before anything is started or tunneled -- the build only needs to know that the API lives at a *relative* `/api/v1` path, not the eventual public URL, so there's no ordering dependency on the tunnel below.

In [ ]:
import os
import subprocess

frontend_build_env = os.environ.copy()
# Relative, not an absolute ngrok URL: once the API and the frontend are
# served from the same port (Step 7), the browser's own fetch calls to
# "/api/v1/..." resolve against whatever host is currently serving the page.
frontend_build_env["REACT_APP_API_BASE_URL"] = "/api/v1"

print("Building frontend (this can take a minute)...")
subprocess.run(
    ["npm", "run", "build"],
    cwd=FRONTEND_DIR,
    env=frontend_build_env,
    check=True,
)
print("Frontend build complete.")

## 6. Install and configure pyngrok

In [ ]:
!pip install -q pyngrok requests

import getpass
from pyngrok import ngrok

ngrok_authtoken = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(ngrok_authtoken)
del ngrok_authtoken  # never keep the token around longer than needed

print("ngrok configured.")

## 7. Start the combined app

Free ngrok accounts are limited to a single simultaneous tunnel/endpoint per agent session -- opening two tunnels (one for the backend API, one for a separate frontend static server, as earlier versions of this notebook did) either fails outright or hands back the same public URL for both, which is the "frontend and backend get the same ngrok URL" problem this version avoids entirely.

The fix is to serve both from the *same* process and port. This cell writes a small wrapper script -- kept in the cloned repo's working copy, not part of the actual project -- that loads the real, unmodified `create_app()` from `backend/app.py` and adds one extra route that serves the frontend's production build (from Step 5) as static files, falling back to `index.html` for the app's own page. `backend/app.py` itself is never changed -- running it directly, per Unit 7 of the lab guide, still works exactly as before.

In [ ]:
import os
import subprocess
import sys
import time

import requests

COMBINED_SERVER_PATH = os.path.join(REPO_PATH, "colab_combined_server.py")
FRONTEND_BUILD_DIR = os.path.join(FRONTEND_DIR, "build")

combined_server_source = f'''"""Colab-only glue: serves the real backend API (unmodified) plus the
frontend's production build from a single port, so this notebook needs only
one ngrok tunnel. Not part of the repository -- generated at runtime.
"""
import os
import sys

sys.path.insert(0, {BACKEND_DIR!r})
from app import create_app  # noqa: E402

from flask import send_from_directory

FRONTEND_BUILD_DIR = {FRONTEND_BUILD_DIR!r}

app = create_app()
# Point Flask's own /static/<path:filename> route (registered by Flask(__name__)
# in app.py, and otherwise unused since backend/static doesn't exist) at the
# frontend build's static/ subfolder -- Create React App serves its JS/CSS from
# exactly that /static/... URL shape, so this needs no extra route.
app.static_folder = os.path.join(FRONTEND_BUILD_DIR, "static")

@app.route("/", defaults={{"path": ""}})
@app.route("/<path:path>")
def serve_frontend(path):
    target = os.path.join(FRONTEND_BUILD_DIR, path)
    if path and os.path.isfile(target):
        return send_from_directory(FRONTEND_BUILD_DIR, path)
    return send_from_directory(FRONTEND_BUILD_DIR, "index.html")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port={BACKEND_PORT!r})
'''

with open(COMBINED_SERVER_PATH, "w") as f:
    f.write(combined_server_source)


def wait_until_up(url, timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return True
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    return False


combined_log_path = os.path.join(REPO_PATH, "combined_server.log")
combined_log = open(combined_log_path, "w")

if "combined_process" in globals() and combined_process.poll() is None:
    print("Combined server already running (PID", combined_process.pid, ")")
else:
    combined_process = subprocess.Popen(
        [sys.executable, COMBINED_SERVER_PATH],
        cwd=REPO_PATH,
        stdout=combined_log,
        stderr=subprocess.STDOUT,
    )
    print("Started combined server (PID", combined_process.pid, ") -- logging to", combined_log_path)

if wait_until_up(f"http://localhost:{BACKEND_PORT}/api/v1/health"):
    print("App is up.")
else:
    print(f"App did not become healthy in time -- check {combined_log_path}")

## 8. Open the ngrok tunnel

Just one: the app (API + frontend, from Step 7) is already listening on `BACKEND_PORT`, so this opens a single tunnel to it.

In [ ]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError


def open_tunnel(port):
    for existing in ngrok.get_tunnels():
        if existing.config["addr"].endswith(f":{port}"):
            return existing
    return ngrok.connect(port, "http")


try:
    app_tunnel = open_tunnel(BACKEND_PORT)
except PyngrokNgrokError as exc:
    raise RuntimeError(
        "Could not open the ngrok tunnel. Check that your authtoken (Step 6) "
        "was entered correctly."
    ) from exc

app_public_url = app_tunnel.public_url
print("App tunnel:", app_public_url)

## 9. Published URL

In [ ]:
print("Open the app here:")
print(f"  {app_public_url}")
print()
print("The app will ask for your Claude API key in the browser -- it is never")
print("entered here, and is used only client-side plus for the single request")
print("it authorizes (SPEC.md Section 6).")

## 10. Shut down (optional)

Run this when you're done to stop the combined server and close the tunnel.

In [ ]:
if "combined_process" in globals() and combined_process.poll() is None:
    combined_process.terminate()
    print("Stopped combined_process (PID", combined_process.pid, ")")

ngrok.kill()
print("ngrok tunnel closed.")